<a href="https://colab.research.google.com/github/Klark-cyber/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Menu detector")

In [ ]:
# Google Colab muhitiga Google Drive-ni ulash uchun maxsus modulni yuklab olamiz
from google.colab import drive
import torchvision.transforms as transforms # imageni tensorga otkazish uchun transformsdan foydalanamiz
from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset # Dataset classini import qildik
import os
import numpy as np

In [ ]:
# Google Drive-ni '/content/drive' papkasiga virtual disk sifatida ulaymiz
# Bu kod ishga tushganda Drive-ga kirish uchun ruxsat so'raydi
drive.mount('/content/drive')

In [ ]:
# Define Dataset Path -> Dataset joylashgan manzilni aniqlash
# Google Drive ichidagi 'food101_dataset' papkasining manzilini o'zgaruvchiga saqlaymiz
DATASET_PATH = '/content/drive/MyDrive/food101_dataset'

# Konsolga dataset manzilini tekshirish uchun chiqarib ko'rsatamiz
print('Dataset_path:', DATASET_PATH)

# Klasslarni o'zgartirish yoki qayta nomlash uchun xarita (mapping) yaratish
# Rasmda bu qism to'liq tugatilmagan, hozircha faqat 'hamburger' kiritilgan
CUSTOM_CLASS_MAPPING = {
    'hamburger': "hamburger",
    "hot_dog": "hot_dog",
    "chocolate_cake": "dessert",
    "cheesecake": "dessert",
    "kebab": "kebab",
    "pilaf": "pilaf"

}

# Model o'rganishi kerak bo'lgan taomlar klaslarining (nomlarining) ro'yxati
CLASSES = ['hamburger', 'hot_dog', 'dessert', 'kebab', 'pilaf']

# Har bir klas nomiga mos ravishda tartib raqami (indeks) biriktiramiz: {'hamburger': 0, 'hot_dog': 1, ...}
# Buning uchun 'enumerate' funksiyasi va Dict Comprehension usulidan foydalanilgan
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}

# Jami nechta klas borligini aniqlaymiz (len funksiyasi ro'yxat elementlari sonini hisoblaydi)
NUM_CLASSES = len(CLASSES)

# Tuzilgan klaslar va ularning indekslari lug'atini (dictionary) konsolga chop etamiz
print(CLASS_TO_IDX)

transform = transforms.Compose([  # compose methodi bu simple ketma ketlikni amalga oshirishni qayt qilissh uchun kerak
    transforms.Resize((224, 224)), # Resuze methodi orqali barcha rasmni bir xil sizega keltirib olamiz
    transforms.ToTensor(), # bu method rasmlarni tensorga otkazib beradi,
    transforms.Normalize(mean=[0,485, 0.456, 0.406], std = [0.229, 0.224, 0.225])   # barcha rasmlarni normal holatga keltiradi (scale+chanelni togri shaklga keltiradi)
])

# 0~255
#0.0~1.0
#0.234
#H, W, Channel => C,H,W | RGB = Red, Green, Blue
#Normalize
#pixel = (pixel - mean)/std



In [ ]:
# Custom Dataset Class
class FoodDataset(Dataset):  # PyTorch-ning Dataset klassidan voris olib, maxsus FoodDataset klassini yaratamiz
    def __init__(self, images, labels, transform=None):  # Klass initsializatori: rasmlar, teglar va transformatsiyalarni qabul qiladi
        self.images = images  # Kelgan rasmlar yo'li (path) ro'yxatini klass xususiyatiga biriktiramiz
        self.labels = labels  # Kelgan teglar (klaslar) ro'yxatini klass xususiyatiga biriktiramiz
        self.transform = transform  # Rasmlarga qo'llaniladigan augmentatsiya/transformatsiyalarni saqlaymiz

    def __len__(self):  # Dataset ichidagi ma'lumotlarning umumiy sonini qaytaruvchi maxsus metod
        print('images_length', len(self.images))  # Konsolga rasmlarning umumiy sonini chop etamiz
        return len(self.images)  # Dataset uzunligi sifatida rasmlar ro'yxati o'lchamini qaytaramiz

    def __getitem__(self, idx):  # Berilgan indeks (idx) bo'yicha bitta rasm va uning tegini qaytaruvchi metod
        img_path = self.images[idx]  # Berilgan indeksga mos keladigan rasm fayli yo'lini aniqlaymiz
        print('image_path', img_path)  # Konsolga yuklanayotgan rasmning manzili (yo'li)ni chop etamiz
        label = self.labels[idx]  # Berilgan indeksga mos keladigan rasm tegini (label) aniqlaymiz
        print('label', label)  # Konsolga rasm tegining qiymatini chop etamiz
        try:  # Rasmni ochishda yuzaga kelishi mumkin bo'lgan xatoliklarni tekshirish bloki
            image = Image.open(img_path).convert('RGB')  # Rasmni ochamiz va uni RGB rang formatiga o'tkazamiz
        except (UnidentifiedImageError, OSError):  # Agar rasm fayli buzilgan yoki ochib bo'lmaydigan bo'lsa xatolikni ushlaymiz
            print(f"Skipping broken image: {img_path}")  # Konsolga buzilgan rasm tashlab ketilgani haqida ogohlantirish chiqaramiz
            return self.getitem((idx + 1) % len(self.images))  # Keyingi indeksdagi rasmni qayta chaqirib, cheksiz sikldan qochish uchun qoldiq olamiz
        if self.transform:  # Agar rasm uchun transformatsiya obyekti mavjud bo'lsa
            image = self.transform(image)  # Rasmni belgilangan transformatsiyalardan (o'lcham, tensorga o'tkazish va h.k.) o'tkazamiz
        return image, label  # Tayyor bo'lgan rasm obyektini va unga mos keladigan tegni qaytaramiz

In [ ]:
# Gather and Split Data
# Ma'lumotlarni yig'ish va qismlarga ajratish uchun sarlavhali izohlar

all_images = []  # Barcha rasmlar yo'li va ularning teglarini juftlik (tuple) ko'rinishida saqlash uchun bo'sh ro'yxat ochamiz
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():  # Klasslar mosligi lug'atidagi har bir asl va yangi klass nomlarini aylanib chiqamiz
    class_path = os.path.join(DATASET_PATH, original_class)  # /content/drive/MyDrive/food101_dataset/hamburger  Har bir klassga tegishli rasmlar joylashgan papkaning to'liq manzilini hosil qilamiz
    print('class_path:', class_path)  # Konsolga tekshirilayotgan klass papkasi manzilini chop etamiz
    if not os.path.exists(class_path):  # Agar ko'rsatilgan klass papkasi diskda mavjud bo'lmasa
        print(f"Warning: {class_path} not found")  # Konsolga papka topilmagani haqida ogohlantirish chiqaramiz
        continue  # Siklning ushbu qadamini tashlab ketib, keyingi klassga o'tamiz
    for img in os.listdir(class_path):  # Klass papkasi ichidagi barcha fayllarni birma-bir aylanib chiqamiz
        if img.endswith(('.jpg', '.jpeg', '.png')):  # Fayl kengaytmasi rasm formatida (.jpg, .jpeg yoki .png) ekanligini tekshiramiz
            full_path = os.path.join(class_path, img)  # /content/drive/MyDrive/food101_dataset/hamburger/100057.jpg Rasm faylining to'liq manzilini (yo'lini) shakllantiramiz
            all_images.append((full_path, CLASS_TO_IDX[mapped_class]))  # (/content/drive/MyDrive/food101_dataset/hamburger/100057.jpg , 1)  Rasmning to'liq manzili va unga mos keladigan indeks raqamini juftlik qilib ro'yxatga qo'shamiz

np.random.shuffle(all_images)  # Ma'lumotlar bir xil turda ketma-ket kelib qolmasligi uchun barcha elementlarni tasodifiy tartibda aralashtiramiz
split = int(0.8 * len(all_images))  # Ma'lumotlarning 80 foizini o'quv (train) to'plami uchun ajratish nuqtasi (indeksini) hisoblaymiz
train_data = all_images[:split]  # Ro'yxatning boshidan to bo'linish nuqtasigacha bo'lgan qismini o'quv ma'lumotlari deb olamiz
val_data = all_images[split:]  # Bo'linish nuqtasidan oxirigacha bo'lgan qismini validatsiya (tekshirish) ma'lumotlari deb olamiz

train_images, train_labels = zip(*train_data)  # O'quv to'plamidagi rasmlar yo'li va teglarini alohida ikkita ro'yxatga ajratib (unzip) olamiz
val_images, val_labels = zip(*val_data)  # Validatsiya to'plamidagi rasmlar yo'li va teglarini alohida ikkita ro'yxatga ajratib (unzip) olamiz

print('all_images:', all_images)  # Konsolga barcha yig'ilgan rasmlar ro'yxatini chop etamiz (kodda biroz xira ko'ringan qism)

dataset = FoodDataset(train_images, train_labels)  # Ajratib olingan o'quv rasmlari va teglaridan foydalanib maxsus Dataset obyektini yaratamiz
print(len(dataset))  # Konsolga yaratilgan dataset ichidagi ma'lumotlarning umumiy sonini chop etamiz
img, lbl = dataset[0]  # Dataset-ning birinchi elementini (rasm va uning tegini) tekshirish uchun o'zgaruvchilarga yuklaymiz